In [2]:
import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [3]:
train = pl.read_csv('../data/BBC News Train.csv')

train

ArticleId,Text,Category
i64,str,str
1833,"""worldcom ex-boss launches defe…","""business"""
154,"""german business confidence sli…","""business"""
1101,"""bbc poll indicates economic gl…","""business"""
1976,"""lifestyle governs mobile choi…","""tech"""
917,"""enron bosses in $168m payout e…","""business"""
…,…,…
857,"""double eviction from big broth…","""entertainment"""
325,"""dj double act revamp chart sho…","""entertainment"""
1590,"""weak dollar hits reuters reven…","""business"""


In [ ]:
df = (
    train
    .select(["Text", "Category"])
    .drop_nulls()
    .with_columns(
        pl.col("Text").cast(pl.Utf8),
        pl.col("Category").cast(pl.Utf8),
    )
    .filter((pl.col("Text").str.len_chars() > 0) & (pl.col("Category").str.len_chars() > 0))
)

X = df["Text"].to_list()
y = df["Category"].to_list()

In [38]:
# split: train/test ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Classes:", sorted(set(y)))

Train size: 1192
Test size: 298
Classes: ['business', 'entertainment', 'politics', 'sport', 'tech']


In [39]:
# LinearSVC
model_svc = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LinearSVC())
])

In [40]:
# LogisticRegression
model_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(max_iter=2000, n_jobs=-1))
])

In [41]:
model = model_svc

model.fit(X_train, y_train)

pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
f1m = f1_score(y_test, pred, average="macro")
f1w = f1_score(y_test, pred, average="weighted")

print(f"Accuracy:     {acc:.4f}")
print(f"F1 macro:     {f1m:.4f}")
print(f"F1 weighted:  {f1w:.4f}")
print("\nClassification report:\n", classification_report(y_test, pred))

labels = sorted(set(y))
cm = confusion_matrix(y_test, pred, labels=labels)

Accuracy:     0.9765
F1 macro:     0.9756
F1 weighted:  0.9765

Classification report:
                precision    recall  f1-score   support

     business       0.97      0.99      0.98        67
entertainment       0.98      0.98      0.98        55
     politics       1.00      0.96      0.98        55
        sport       0.97      1.00      0.99        69
         tech       0.96      0.94      0.95        52

     accuracy                           0.98       298
    macro avg       0.98      0.97      0.98       298
 weighted avg       0.98      0.98      0.98       298



In [42]:
model = model_lr

model.fit(X_train, y_train)

pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
f1m = f1_score(y_test, pred, average="macro")
f1w = f1_score(y_test, pred, average="weighted")

print(f"Accuracy:     {acc:.4f}")
print(f"F1 macro:     {f1m:.4f}")
print(f"F1 weighted:  {f1w:.4f}")
print("\nClassification report:\n", classification_report(y_test, pred))

labels = sorted(set(y))
cm = confusion_matrix(y_test, pred, labels=labels)

Accuracy:     0.9664
F1 macro:     0.9656
F1 weighted:  0.9663

Classification report:
                precision    recall  f1-score   support

     business       0.94      0.99      0.96        67
entertainment       0.98      0.96      0.97        55
     politics       1.00      0.95      0.97        55
        sport       0.96      1.00      0.98        69
         tech       0.96      0.92      0.94        52

     accuracy                           0.97       298
    macro avg       0.97      0.96      0.97       298
 weighted avg       0.97      0.97      0.97       298

